# TritFlow：オープンソースPDK TR-1umで、三値のHDLからGDSへ

Einosuke Okazaki (岡崎 英之介)<br>
Information and Communication Engineering Course, Department of Monozukuri Engineering, Tokyo Metropolitan College of Industrial Technology (KOSEN), Tokyo, Japan

**実行環境**

- 所要時間：新しいColabランタイムでの「Run all」実行時、セットアップを含めて約6〜8分
- Python：3.12
- PDK：OpenSUSI TR-1um（commit 5d963db）
- KLayout：0.30.6
- ngspice：42
- Pythonパッケージ：先頭のセットアップセルにて固定バージョンを自動導入

TritFlow（トリットフロー）は、3つの電圧レベルを直接扱うCMOSスタンダードセルを用い、三値HDLからGDSレイアウトの生成までをオープンソースPDK「OpenSUSI TR-1um」上で実現する完全なEDA設計フローです。本ノートブックでは、三値HDL記述からの論理合成、2層メタルによる自動配置配線、GDSII出力、PDK公式デッキによるDRC/LVS検証、寄生容量を考慮したポストレイアウトSPICEシミュレーション、そして三値固有の静的タイミング解析（STA）までを一貫して対話的に実行します。

![fig_flow_v2](https://raw.githubusercontent.com/PenguinEino/tritflow/6c56455e09d1ac06f73dfad8fc37c07e5210197f/figures/out/fig_flow_v2_ja.png)

TritFlowが扱うのは**平衡三値（Balanced Ternary）**です。各桁（トリット）は −1・0・+1 の3値をとり、桁の重みは3のべき乗（1, 3, 9, …）となります。回路上の電圧としては、それぞれ −5 V・0 V・+5 V に対応します。平衡三値は、1958年にモスクワ大学で開発された初期の三値計算機「[Setun](https://doi.org/10.1007/978-3-642-22816-2_10)」でも採用された歴史ある表現形式です。近年では、重みをこの三値（−1・0・+1）に制約した大規模言語モデル「[BitNet b1.58](https://doi.org/10.48550/arXiv.2402.17764)」の登場により再び大きな注目を集めています。線形層における重みと8ビット入力との積は「入力そのまま」「符号反転」「ゼロ」のいずれかとなるため、乗算器を用いず加減算のみで積和演算（GEMM）を実行可能です。三値のデータを中間の二値変換を挟まず、三値の電圧のままダイレクトに処理する専用ハードウェアは、極めて高い電力効率を達成する可能性を秘めています。

しかし現在、三値演算を担っているハードウェアのほぼすべては、従来の二値論理回路上でのエミュレーションです。デバイスレベルではCNTFET（[Lin et al., 2011](https://doi.org/10.1109/TNANO.2009.2036845)）、メモリスタ（[Wang et al., 2021](https://doi.org/10.1109/TCSI.2020.3027693)）、新規CMOS構造（[Jeong et al., 2019](https://doi.org/10.1038/s41928-019-0272-8)）など多様な三値素子が研究されているものの、設計ツール環境は[ゲート回路網の生成](https://doi.org/10.1109/ACCESS.2025.3597293)にとどまるか、あるいは三値信号を複数の二値信号線に符号化して二値EDAを通すアプローチ（[MRCS](https://github.com/aiunderstand/MixedRadixCircuitSynthesis/tree/0865cee1f49fe9a803b4910b570f4e96e514c61b)、[Takahe](https://github.com/Zaneham/takahe/blob/e7123ea60269922acb283d576678a3aabe51f4f4/designs/setun70_physical.sv)、[Ankhdjet](https://doi.org/10.48550/arXiv.2608.26206)）に限られていました。商用ファブで製造可能なオープンソースPDK上で、3つの物理電圧レベルを直接扱うCMOSセルを用いて、HDLからGDSまでを完全自動で通すフィジカル設計フローは、我々の知る限り前例がありません。

本ノートブックでは、日本の商用半導体ファブ（東海理化）が公開したオープンソースPDK「[OpenSUSI TR-1um](https://github.com/OpenSUSI/TR-1um/tree/5d963db9d44e192d37a0489c208bfccf2e2f690c)」を採用し、この三値設計フローを実証します。TritFlowのセルライブラリは、[著者が設計・テープアウトした三値論理LSI](https://github.com/PenguinEino/opensusi-tr10-balanced-ternary-logic/tree/8a60b97868cad008ff21fefca5da13dcd06063f8)（2026年、第1回OpenSUSI-MPWシャトル「OpenSUSI-TR10」）の実績ある回路トポロジーと素子寸法をそのままスタンダードセル化したものです。三値専用HDL「TritVerilog」の記述から論理合成、配置配線、DRC/LVS検証、ポストレイアウトSPICEシミュレーションまでを一つの流れるようなパイプラインとして完結させています。

OpenSUSI-TR10で製造した実チップは2027年1月に納品予定であり、本提案が採択された暁には、2月のISSCC会場においてTritFlowのデモとともに、三値セルのシリコン実測結果を世界で初めて報告します。さらにTritFlowを用いて設計した大規模な三値マクロ回路は、次回のOpenSUSIシャトルにて製造を委託します。「設計・製造・実測・学会報告」のサイクルを一気通貫で完結させることで、TR-1um PDKがIEEE SSCS Code-a-Chipの掲げる3つの核となる価値を実現するに足る実力を持つことを証明し、日本におけるオープン半導体エコシステムと設計の民主化を力強く前進させます。

**再現性（Reproducibility）**　後に続く研究者や技術者が、TritFlowの動作原理を隅々まで理解し、自らの手で再構築・検証できるように設計されています。そのためにセルの内部回路、論理合成アルゴリズム、配置配線の手法はいずれも、コードを読んで明快に追尾できる直感的な見通しの良さを重視しました。同時に、PDK公式のDRC/LVSデッキを完全にパスし、抽出後SPICEで動作確認された本物のシリコン品質のGDSを出力します。ノートブック内の各セルは各処理段が「なぜそのように動作するのか」を物理的・理論的な背景とともに説明し、読者がその場でパラメータを変えて即座に検証できるように構成されています。さらに、セル自体が実際にテープアウトされた回路と同一であるため、シミュレーション結果を将来の実測データと直接突き合わせることが可能です。

**再利用性（Reusability）**　ノートブックで得た理解をもとに、TritFlowを土台として読者独自の研究や設計へと容易に発展させることができます。TritVerilogの記述を書き換えるだけで、同一の自動フローによって検証済みの新しい三値コアレイアウトを瞬時に再生成できます。後半では、4項の内積積和回路（dot4）をMPWシャトルの標準フレームおよび入出力パッドと結合したフルチップ・レイアウトまで自動生成します。すべてのアルゴリズムが明瞭なPythonモジュールとして分割されているため、新しい三値ゲートの追加や、論理合成・配置配線アルゴリズムの拡張も自在です。セルライブラリ、EDAパイプライン、本ノートブックはすべてオープンソース（Apache-2.0）で公開されており、次世代の三値コンピューティング研究の強固な出発点となります。

**教育性（Educational Value）**　この「理解から発展へ」の道筋は、そのまま次代を担う学生やエンジニアにとって最高の教材となります。すべての工程がGoogle Colab上で対話的に実行され、各段階で回路網やレイアウト、電圧波形がどのように変化したのかを豊かな可視化ウィジェットと詳細な数値データで示します。二値CMOSの基本とVerilogの初歩を理解している読者であれば、HDLの1行から始めてシリコン製造用GDSが完成するまでの全貌を、自らの手を動かしながら体系的に体験できます。単なる理論のシミュレーションにとどまらず、実際の半導体製造とシリコン実測に直結した本物の集積回路設計の醍醐味を体感してください。

In [ ]:
#@title 準備 { display-mode: "form" }

import os, sys, json, time, subprocess, importlib.util, re
from pathlib import Path
SOURCE_URL = "https://github.com/PenguinEino/tritflow.git"
SOURCE_REF = os.environ.get("TRITFLOW_REF", "6c56455e09d1ac06f73dfad8fc37c07e5210197f")
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

def git(*args, progress=None):
    """Run git; with ``progress``, show git's own counter on one redrawn line."""
    if progress is None:
        return subprocess.check_output(["git", "-C", str(ROOT), *args], text=True).strip()
    from IPython.display import display, HTML
    style = "font:13px/1.6 ui-monospace,Menlo,Consolas,monospace;white-space:pre"
    line = display(HTML(f'<div style="{style}">⠿ {progress}</div>'), display_id=True)
    process, text, started = subprocess.Popen(["git", *args], stderr=subprocess.PIPE, text=True), "", time.monotonic()
    for chunk in iter(lambda: process.stderr.read(64), ""):
        text += chunk
        parts = [p.strip() for p in re.split(r"[\r\n]", text)[:-1] if p.strip()]
        if parts:
            line.update(HTML(f'<div style="{style}">⠿ <b>{progress}</b>  <span style="color:#8a8f98">{parts[-1][:80]}</span></div>'))
    if process.wait():
        raise RuntimeError(f"git {' '.join(args[:2])} failed:\n{text[-2000:]}")
    line.update(HTML(f'<div style="{style}"><span style="color:#1a9c4a">✓</span> <b>{progress}</b>  '
                     f'<span style="color:#8a8f98">{time.monotonic()-started:.1f}s</span></div>'))

if IN_COLAB:
    ROOT = Path("/content/tritflow")
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF):
        raise ValueError("TRITFLOW_REF must be a full 40-character commit")
    if not ROOT.exists():
        git("clone", "--progress", "--no-checkout", "--depth", "1", "--filter=blob:none",
            SOURCE_URL, str(ROOT), progress="GitHubからソースを取得")
        git("-C", str(ROOT), "fetch", "--progress", "--depth", "1", "origin", SOURCE_REF,
            progress=f"コミット {SOURCE_REF[:7]} を取得")
        git("-C", str(ROOT), "switch", "--detach", SOURCE_REF, progress="ファイルを展開")
    if git("rev-parse", "HEAD") != SOURCE_REF or git("status", "--porcelain", "--untracked-files=no"):
        # A runtime left by another version of this notebook: move it to the pinned commit and
        # drop the modules the kernel already imported from the old checkout.
        git("-C", str(ROOT), "fetch", "--progress", "--depth", "1", "origin", SOURCE_REF,
            progress=f"コミット {SOURCE_REF[:7]} に切り替え")
        git("reset", "--hard", SOURCE_REF)
        for name, module in list(sys.modules.items()):
            if str(getattr(module, "__file__", "") or "").startswith(str(ROOT)):
                del sys.modules[name]
else:
    candidates = [Path(os.environ.get("TRITFLOW_ROOT", ".")), Path.cwd(), Path.cwd().parent]
    ROOT = next((p.resolve() for p in candidates if (p/"tritflow").is_dir() and (p/"notebook").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("run from the TritFlow checkout or set TRITFLOW_ROOT")
sys.path[:0] = [str(ROOT), str(ROOT/"notebook")]
from live import step, run_stages, set_language as set_notebook_language
set_notebook_language("ja")
from runtime_support import SETUP_SCHEMA, setup_identity, check_runtime, print_tool_versions
runtime = ROOT/"notebook/runtime.json"
if IN_COLAB and (not runtime.is_file()
                 or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SHA256") != setup_identity(ROOT)):
    os.environ["TRITFLOW_NOTEBOOK_LANGUAGE"] = "ja"
    run_stages(["bash", str(ROOT/"notebook/colab_setup.sh"), "--source", str(ROOT)])
if runtime.is_file():
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
with step("ツールを確認") as s:
    check_runtime()
    tools = print_tool_versions()
    s.done(" · ".join(re.search(r"(KLayout [\d.]+|ngspice-\d+)", v)[1] for v in tools.values()))
BUILD = Path(os.environ.get("TRITFLOW_NOTEBOOK_BUILD", "/content/build" if IN_COLAB else ROOT/"build/notebook")).resolve()

from live import need, provides, install_error_view, LayoutView
from views import pick_input, stimulus, bus_picker, cell_counts, schematic_view, scope_figure, timing_table, wire_table, delay_table, transient_conditions, dc_conditions, note
from IPython.display import HTML, display
import plotly.io as pio
pio.renderers.default = "colab"
from inverter_views import inverter
from pnr_views import replay
from cell_views import closure_view, cell_tables, cell_row
from tritflow import CELLS_DIR
from tritflow.sim.sizing import sweep_dimensions, load_grid
from tritflow import flow
from tritflow.viz.layout import gds_viewer
from tritflow.schematic import set_language
set_language("ja")
install_error_view()
for name, cell in {"source": "1-1 HDL", "net": "1-2 合成する", "lay": "1-3 配置配線する",
                   "lvs": "1-4 DRCとLVS", "post": "1-5 ポストレイアウトシミュレーション",
                   "dot4": "3-8 論理合成する", "dot4_layout": "4-1 配置配線する",
                   "dot4_lvs": "5-1 DRCとLVS", "dot4_extracted": "5-2 抽出と配線容量", "dot4_post": "5-3 ポストレイアウトシミュレーション",
                   "dot_scaling": "6-1 積和の規模と面積",
                   "reuse_source": "6-4 HDLを変更する"}.items():
    provides(name, cell)


## 1. 最初の回路：`a & ~a` をGDSにする

本章では、極めてコンパクトでありながら三値論理の真髄を如実に表す回路 `a & ~a` を題材に、TritFlowの全体フロー（HDL記述から論理合成、配置配線、DRC/LVS、ポストレイアウトSPICEシミュレーションまで）を一気通貫で体験します。各処理段の詳細な仕組みは2章以降で順を追って解説します。

二値論理において、入力 a とその反転の論理積 a & ~a は「矛盾律」により常に偽（0）となり、回路としてはグラウンドに直結した定数へと縮退してしまいます。しかし、平衡三値論理ではそうはなりません。

下表に示すように、平衡三値の論理積（AND）は「2つの値のうち小さい方（min）」として定義され、符号反転（NOT, `~`）は正負を反転する操作（+1 ↔ −1, 0 ↔ 0）です。入力 a が +1 や −1 のときは出力が最小値の −1 となりますが、a = 0（0 V）が与えられた場合、反転した ~a も 0 となるため、0 & 0 の出力は **0（0 V）** になります。すなわち `a & ~a` は定数ではなく、a = 0 の入力に対して固有の中間電圧を出力する、物理的な実体を持った立派な三値回路なのです。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@6c56455e09d1ac06f73dfad8fc37c07e5210197f/notebook/assets/tables/and.svg" width="660" alt="and">

### 1.1 HDL記述（TritVerilog）

まず、この回路をTritFlow専用のHDL「TritVerilog」で記述します。三値の信号線は型 `trit` で宣言し、代入文 `assign` で論理積 `&` と反転 `~` を記述します。

In [ ]:
# 1-1 HDL
source = """
module first(input trit a, output trit y);
  assign y = a & ~a;
endmodule
"""

### 1.2 論理合成

HDL記述を解析し、三値スタンダードセルの回路網へと変換します。TritFlowの論理合成器は、三値の論理式を三値セル（ここでは反転を行うINVと、否定論理積を行うNAND）へとダイレクトに写像します。同時に、全入力パターン（ここでは a = −1, 0, +1 の3通り）を網羅的に評価し、合成後の回路網が元のHDLと数学的に完全一致することを自動検証します。

In [ ]:
# 1-2 合成する
need("source", why="合成するHDLは、1-1に書いてあります。")
with step("HDLを読み込んで合成") as s:
    net = flow.loads(source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in net.report()["cells"].items())
           + (" · 全入力で等価" if net.equivalence["passed"] else " · 等価でない"))
bus_picker(net)

### 1.3 自動配置配線（P&R）

合成されたゲートの接続関係（ネットリスト）をもとに、TR-1um PDKの幾何ルールに沿ってセルを自動配置し、2層の配線層（M1/M2）で結線します。セル間を水平に貫く3本の電源レール（上からVDD=+5 V、VMID=0 V、VSS=−5 V）が互いに隙間なく密着するようにセルが並べられ、信号線が短距離で整然と配線されます。

In [ ]:
# 1-3 配置配線する
need("net", why="配置配線する回路 net は、1-2で合成します。")
with step("配置配線", view=LayoutView()) as s:
    lay = net.place_and_route(output_dir=BUILD/net.top, seed=0)
    width, height = lay.stats["die_size_um"]
    s.done(f"{lay.stats['instance_count']}セル · {width:.0f} × {height:.0f} µm · 配線 {lay.stats['routed_nets']}本",
           picture=replay(lay))

### 1.4 物理検証（DRC / LVS）

完成したレイアウトが半導体ファブの製造基準を満たしているかを、PDK公式のKLayoutデッキを用いて検証します。設計規則違反がないかを調べるDRC（Design Rule Check）と、レイアウトから抽出したトランジスタ接続が論理合成ネットリストと一致しているかを照合するLVS（Layout Versus Schematic）を自動実行します。いずれもエラーゼロ（Pass）となることを確認してください。

In [ ]:
# 1-4 DRCとLVS
need("lay", why="DRCとLVSは、1-3で作ったGDSを調べます。")
with step("DRC") as s:
    drc = lay.drc()
    s.done(f"違反 {drc['violations']}件")
with step("LVS") as s:
    lvs = lay.lvs()
    s.done("一致" if lvs["matched"] else "不一致")
gds_viewer(lay.gds_path)

### 1.5 ポストレイアウトSPICEシミュレーション

レイアウトの幾何形状から実際の配線寄生容量を抽出し、トランジスタの非線形特性を含む本物のSPICEネットリストを生成して、ngspiceによる過渡解析を実行します。入力 a を −5 V → 0 V → +5 V と変化させたとき、各内部ノードと出力ノードの電圧が目標レベル（−5 V・0 V・+5 V）の許容判定帯（±0.5 V）に正確に整定する様子をインタラクティブな波形で確認できます。

In [ ]:
# 1-5 ポストレイアウトシミュレーション
need("lvs", why="シミュレーションは、1-4でLVSが一致したGDSから回路を取り出して動かします。")
with step("抽出と配線容量") as s:
    extracted = lay.extract()
    signals = [n for n in extracted.capacitance["nets"] if n not in ("VDD", "VMID", "VSS")]
    s.done(f"信号{len(signals)}本に配線容量を追加")
with step("ngspice") as s:
    post = extracted.simulate(stimulus(net), load=100e-15, hold_s=1e-6, step_s=2e-9,
                              edge_s=1e-9, temperature_c=27)
    s.done("全区間で ±0.5 V の帯に入った" if post["measurements"]["passed"] else "帯に入らない区間がある")
pick_input(net, post)
transient_conditions(post, load_f=100e-15)

これで、三値HDLから製造可能なGDSの生成と動作検証までの一巡が完了しました。全体のパイプラインと各段の入出力関係を振り返ると、次の図のようになります。

![fig_flow_map](https://raw.githubusercontent.com/PenguinEino/tritflow/6c56455e09d1ac06f73dfad8fc37c07e5210197f/figures/out/fig_flow_map_ja.png)

## 2. セル

1章で動かした回路は、INVとNANDという2種類のスタンダードセルで構成されていました。本章では、TritFlowを支える三値スタンダードセルライブラリの物理構造と理論的根拠を詳しく見ていきます。まず最も基本となるINV（反転ゲート）を例に、CMOSプロセス上で3つの異なる電圧レベルをいかにして自律的に生成するかを物理レベルで確かめます。次に、どのようなセルの組み合わせがあれば任意の三値論理関数を漏れなく実現できるのか（機能的完全性）を数学的に証明し、最後に、テープアウト済みの実績ある回路を自動配置配線可能なスタンダードセルへと昇華させたレイアウト設計について解説します。

### 2.1 INV：3つの電圧レベルを自律生成する

二値CMOSインバータとの決定的な違いは、中間の論理値「0」（0 V）を出力するために、入力0 V時に上下のMOSトランジスタを**同時にオン**させ、内部の分圧比によって中間の電圧レベルを生成する点にあります。

もし0 Vレール（VMID）からスイッチ（トランスファーゲート等）を用いて直接出力しようとすると、入力が0 Vであることを判定するための追加回路が必要となり、素子数と面積が大幅に増大してしまいます。これに対しTritFlowのINVは、電源レールVDD（+5 V）とVSS（−5 V）の間にPMOS、2本のP型拡散抵抗（PDKの素子名RR）、およびNMOSを直列に接続し、2本の抵抗の中点から出力Yを取り出す、極めて簡潔で堅牢なトポロジーを採用しています。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@6c56455e09d1ac06f73dfad8fc37c07e5210197f/notebook/assets/tables/inv.svg" width="650" alt="inv">

回路の動作は次の3つの領域に分かれます。

1. **入力 A = −5 V（論理値 −1）のとき**：PMOSのゲート・ソース間電圧 V<sub>GS</sub> が −10 V となって強くオンし、NMOSの V<sub>GS</sub> は 0 V となってオフします。出力Yは上側の電源レールVDDにプルアップされ、**+5 V（論理値 +1）** を出力します。
2. **入力 A = +5 V（論理値 +1）のとき**：NMOSの V<sub>GS</sub> が +10 V となって強くオンし、PMOSの V<sub>GS</sub> は 0 V となってオフします。出力Yは下側の電源レールVSSにプルダウンされ、**−5 V（論理値 −1）** を出力します。
3. **入力 A = 0 V（論理値 0）のとき**：PMOSの V<sub>GS</sub> は −5 V、NMOSの V<sub>GS</sub> は +5 V となり、両方のトランジスタが同時に導通します。その結果、VDDからVSSへと電流が流れ、2本の直列拡散抵抗による分圧によって正確な **0 V（論理値 0）** が自律的に生成されます。

この0 V出力状態では、VDDとVSSの間に貫通静的電流（DCバイアス電流）が流れ続けます。拡散抵抗RRは、この静的電力を許容範囲に抑える電流制限抵抗としての重要な役割を担っています。

#### 手計算による設計とSPICEシミュレーションの照合

素子寸法の選定にあたっては、中間レベルの出力精度、伝搬遅延、セル面積、そして静的消費電力のトレードオフを厳密に考慮しました。TritFlowのセル寸法は、2026年にテープアウトされた実チップ「OpenSUSI-TR10」の採用寸法と完全に一致させています。

- **判定基準**：中間レベルの十分な雑音余裕を確保するため、入力電圧が −1〜+1 V の範囲内にあるとき、出力電圧が 0 V ±0.5 V の許容判定帯に留まること。
- **ゲート長 L**：駆動能力の最大化とセル面積の縮小のため、プロセスの最小ルールである 1 µm を採用。
- **PMOSとNMOSのチャネル幅 W**：入力0 V時の出力電圧は、上側のオン抵抗と拡散抵抗の和 R<sub>u</sub> = r<sub>on,P</sub> + R<sub>RP</sub> が、下側の R<sub>d</sub> = r<sub>on,N</sub> + R<sub>RN</sub> と等しいときに0 Vとなります。正孔の移動度は電子より低いため、PMOSの幅（W<sub>P</sub> = 13.5 µm）をNMOSの幅（W<sub>N</sub> = 5 µm）の2.7倍に広げることで、両者のオン抵抗を近づけ、DCオフセットを極小化しました。
- **拡散抵抗RRの幅と長さ**：幅は最小ルールの W = 2.8 µm、長さはチップ全体の静的電流を電源配線やビアの許容電流内に抑え込むため、L = 30 µm としました。

ここで、手計算による理論値（Murmann流の手計算モデル）とSPICEシミュレーション結果を比較してみましょう。
PDKのゼロバイアス抵抗式から各拡散抵抗値は R<sub>RP</sub> = R<sub>RN</sub> ≃ 10.835 kΩ と算出されます。また、入力0 VにおけるMOSの線形領域近似からオン抵抗を見積もると、r<sub>on,P</sub> ≃ 0.492 kΩ、r<sub>on,N</sub> ≃ 0.399 kΩ となります。
注目すべきは、拡散抵抗RRの値（10.835 kΩ）がMOSのオン抵抗（約0.4〜0.5 kΩ）の**20倍以上**（PMOS側22.0倍、NMOS側27.2倍）もの大きさを占めている点です。これにより、製造ばらつきや温度変動に起因するMOS特性の変動は大幅に希釈され、0 Vの出力レベルは「同一レイアウトで近接配置された一対の受動抵抗の比精度」によって極めて安定して保たれます。

上下の合成抵抗から、手計算による出力電圧 V<sub>Y</sub> および静的電流 I は次のように導かれます。

$$V_Y = V_{SS} + (V_{DD}-V_{SS})\frac{R_d}{R_u+R_d} \simeq -0.021\ \mathrm{V},\qquad I = \frac{V_{DD}-V_{SS}}{R_u+R_d} \simeq 0.443\ \mathrm{mA}$$

これに対し、高次効果を含むngspiceのDC解析結果は **V<sub>Y</sub> = −0.040 V**、**I = 0.407 mA** となります。手計算とのわずかな差は、拡散抵抗の電圧・基板バイアス効果（空乏層の広がりによる実効抵抗の増加）と、トランジスタの弱反転・飽和領域遷移に伴う非線形性を手計算では省略していることによります。いずれにしても、出力オフセットは −40 mV に過ぎず、目標である ±0.5 V（±500 mV）の許容帯に対して1桁以上小さく、十分な雑音余裕度を誇ります。

下のセルでは、テープアウト寸法を中心とする90通りのトランジスタ・抵抗寸法スイープ結果を可視化します。

In [ ]:
# 2-1 INVのDC解析と寸法
with step("INVのDC解析") as s:
    spice = sweep_dimensions("INV", [], workdir=BUILD/"inv")
    zero = next(r for r in spice["rows"][0]["dc"] if r["input_trits"] == [0])
    s.done(f"入力 0 V：Y = {zero['output_v']['Y']:+.3f} V · {abs(zero['rail_current_a']['VDD'])*1e3:.3f} mA")
with step("寸法の記録を照合") as s:
    grid = load_grid("INV")
    s.done(f"{len(grid['rows'])}通り · 回路・モデル・解析コードが一致")
from predictions import divider, rail_power
from tritflow import PDK_ROOT
from style import figure_css, table_css
inv_hand = divider(grid["rows"][grid["adopted_index"]]["dimensions"], PDK_ROOT)
upper, lower = inv_hand["upper"], inv_hand["lower"]
resistances = [upper["rr_ohm"], upper["ron_ohm"], lower["ron_ohm"],
               upper["total_ohm"], lower["total_ohm"]]
comparison = [("V<sub>Y</sub> (V)", inv_hand["output_v"], zero["output_v"]["Y"], "+.3f"),
              ("I (mA)", inv_hand["current_a"]*1e3, abs(zero["rail_current_a"]["VDD"])*1e3, ".3f"),
              ("P (mW)", inv_hand["power_w"]*1e3,
               rail_power(zero, grid["conditions"]["supplies_v"])*1e3, ".3f")]
display(HTML('<style>'+figure_css('.inv-estimate')+table_css('.inv-estimate')+'</style>'
    + '<div class="inv-estimate"><table><tr>'
    + ''.join(f'<th>{label}<small>kΩ</small></th>' for label in
              ('R<sub>RR</sub>', 'r<sub>on,P</sub>', 'r<sub>on,N</sub>', 'R<sub>u</sub>', 'R<sub>d</sub>'))
    + '<th>R<sub>RR</sub>/r<sub>on,P</sub></th><th>R<sub>RR</sub>/r<sub>on,N</sub></th></tr><tr>'
    + ''.join(f'<td>{r/1e3:.3f}</td>' for r in resistances)
    + f'<td>{upper["rr_ohm"]/upper["ron_ohm"]:.1f}</td><td>{lower["rr_ohm"]/lower["ron_ohm"]:.1f}</td></tr></table>'
    + '<table style="margin-top:1em"><tr><th>入力0 V</th><th>抵抗による近似</th><th>SPICE</th></tr>'
    + ''.join(f'<tr><td>{label}</td><td>{estimate:{fmt}}</td><td>{measured:{fmt}}</td></tr>'
              for label, estimate, measured, fmt in comparison)
    + '</table></div>'))
inverter(spice, grid)
dc_conditions()

### 2.2 機能的完全性：任意の三値論理を網羅する条件

論理合成の使命は、任意のHDL記述を、あらかじめ用意された限られた種類のスタンダードセルの組み合わせへと自動変換することです。これが可能であるためには、ライブラリが**機能的に完全（Functionally Complete）**でなければなりません。二値論理では、NANDゲートまたはNORゲートが1種類あれば、反転・論理積・論理和を含むすべての論理関数を構成できることが知られています。

三値論理の世界において、TritFlowは後述する**NANY**ゲートと、正の定数を出力する**TIEP**セルの、わずか**2種類のプリミティブ**によって機能的完全性を満たすことを証明しました。

- **NANY**：TritFlowで定義した2入力三値ゲート。2つの入力の和を −1〜+1 の範囲にクランプ（飽和）したのち、符号を反転します。すなわち NANY(a, b) = −sat(a + b) です。
- **sat**：入力値を −1〜+1 の区間に切り詰める飽和関数：sat(x) = max(−1, min(+1, x))。
- **TIEP / TIEZ / TIEN**：それぞれ +1, 0, −1 の固定電位を出力する定数セル。二値のタイセル（TIEHI/TIELO）に相当します。完全性の観点からは、+1 を供給するTIEP（または −1 を供給するTIEN）が1種類あれば十分です。

この完全性の理論的支柱となるのが、ポーランドの論理学者 Jerzy Słupecki による判定定理（Słupecki, 1939）です。この定理によれば、三値関数の集合は次の2条件を満たすとき、かつそのときに限り機能的に完全となります。

1. **すべての1入力関数（単項演算）を合成可能であること**：入力 a の取り得る値（−1・0・+1 の3通り）それぞれに対して出力を独立に選べるため、1入力関数は全部で 3 × 3 × 3 = 27通り存在します。これら27種類すべてを生成できなければなりません。
2. **本質的な2入力関数を少なくとも1つ生成可能であること**：2つの入力 a, b の双方に依存し（片方を固定しても、もう片方の変化によって出力が変化し得る）、かつ出力として −1・0・+1 の3値すべてを取り得る2入力関数が存在すること。

条件2は、NANY自身が直ちに満たします。NANY(a, 0) = −a であるため a の変化が出力に直接反映され、b についても同様に対称です。また出力値も −1・0・+1 のすべてをとります。

条件1については、入力変数 a と定数 +1 のみを出発点として、既知の関数の出力をNANYの入力に次々と再帰的に接続していき、27種類すべての単項関数が生成されることを閉包計算（Closure check）によって網羅的に確認します。下表は、NANYの論理段数（深さ）とともに生成される関数の推移を示しています。

In [ ]:
# 2-2 NANYと +1 の定数から作れる1入力関数
from completeness import generations, surjective_and_depends_on_both, nany
found = generations(binary=(nany,), constants=(1,))
print(f"作れた1入力関数：{len(found)} / 27、最も深いもので {max(found.values())} 段")
print("NANYは2つの入力の両方に依存し、出力に−1・0・+1のすべてが現れる：", surjective_and_depends_on_both(nany))
closure_view(found)

なぜNANY単体では完全にならず、定数セル（TIEP）が必要なのでしょうか？
それは、NANY(0, 0) = −sat(0 + 0) = 0 という性質があるためです。もしすべての入力が論理値0（0 V）であった場合、NANYをいくら複雑に何千段接続しようとも、回路網のすべてのゲート出力は永久に0のまま固定されてしまいます。これでは、入力が0のときに +1 や −1 を出力する関数はおろか、定数 +1 や −1 そのものすら作ることができません。この「原点固定の呪縛」を破るために、0以外の定数を供給するTIEPが1つだけ不可欠となるのです。残る定数は NANY(+1, +1) = −1、および NANY(+1, −1) = 0 として作れるため、外部定数は1種類で事足ります。TritFlowの論理合成アルゴリズムは、この完全性を基盤として、任意の真理値表をゲート網へと確実に置き換えます。

### 2.3 スタンダードセルライブラリの一覧

TritFlowが提供するスタンダードセルの一覧を以下に示します。INV、NAND、NOR、NANYの4つの主要論理ゲートは、OpenSUSI-TR10で実際にテープアウトされたシリコン回路そのものです。NANDやNORは完全性の観点からは必須ではありませんが、回路網の論理段数を削減し、面積と遅延を最適化するために用意されています。これらに加え、高ファンアウト信号を駆動するためのバッファBUF（INVの2段直列接続）と、定数セル（TIEP/TIEZ/TIEN）を備えています。

各セルの回路図記号において、出力端の白丸は符号反転（NOT）を意味し、NANDとNORは二値論理でおなじみの記号形状を踏襲しています。NANYは総和器（Σ）と飽和・反転を組み合わせた独自のシンボルで表記します。

In [ ]:
# 2-3 セルの真理値表
cell_tables(CELLS_DIR/"out", ROOT/"notebook/assets/schematics")

### 2.4 スタンダードセルのレイアウト構造

自動配置配線（P&R）を実現するためには、すべてのセルが厳格な幾何学的ルールに従って設計されていなければなりません。TritFlowのセルライブラリは、行の高さ（セルハイト）を全セル共通の **132 µm**、配置配線格子ピッチを **6 µm** に統一しています。

セル内部には、上から順に **VDD（+5 V, 幅 8 µm）**、**VMID（0 V, 幅 4 µm）**、**VSS（−5 V, 幅 4 µm）** の3本の第1層メタル（M1）電源レールが水平に貫通しています。セルを横に隙間なく突き当てて並べるだけで、電源レールとNウェル、基板コンタクトが自動的に連続して接続されます。信号入力ピンおよび出力ピンは、上層の第2層メタル（M2）配線が直交してビア（V1）で無理なくアクセスできるよう、配線格子に沿ったM1ランドとして配置されています。また、セルは通常配置（R0）に加えて左右反転（MY）での配置も完全に許容されています。

In [ ]:
# 2-4 スタンダードセル
import json
cell_row(json.loads((CELLS_DIR/"out/library.json").read_text()), CELLS_DIR/"out")

## 3. HDLと論理合成

2章では、TritFlowのセルライブラリが任意の三値論理を網羅できる機能的完全性を備えていることを確認しました。本章では、三値回路をアルゴリズム的に記述する専用言語「**TritVerilog**」と、その記述をスタンダードセルの接続網へと最適変換する論理合成エンジンについて見ていきます。TritVerilogは、業界標準のVerilog HDLの構文体系を三値へと自然に拡張したものであり、直感的な文法で大規模な三値デジタル回路を設計できます。

### 3.1 値と記数法：平衡三値の数学的エレガンス

TritVerilogが扱う最小単位の信号値（トリット：trit）は −1・0・+1 の3値であり、回路上の −5 V・0 V・+5 V に対応します。論理判断の文脈では、それぞれ「偽（False）」「不定（Unknown / Undecided）」「真（True）」を意味します。ここでいう「不定」とは、二値論理シミュレータの未初期化値XやハイインピーダンスZのようなエラー状態ではなく、**物理的な0 V電位として回路内に実在する、完全に有効かつ安定した第3の論理状態**である点に注意してください。

複数のトリットを束ねたベクトル（例：`trit [1:0] a;`）は、**平衡三値（Balanced Ternary）**による整数値を表します。下位から数えて第 i 桁の重みは 3<sup>i</sup> となります。数値リテラルでは、値 −1 を記号 `T`（あるいは小文字 `t`）で表記します。たとえば `2'tT1` は、−1·3 + 1 = −2 を表します。2トリットあれば −4 から +4 までの計9通りの整数を過不足なく表せます。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@6c56455e09d1ac06f73dfad8fc37c07e5210197f/notebook/assets/tables/numbers.svg" width="566" alt="numbers">

平衡三値記数法には、従来の二進数表現（2の補数表現など）にはない極めて洗練された特長があります。

- **符号ビットが不要**：−1 を対称に含むため、特別な符号ビットを設けることなく、正数・ゼロ・負数を単一の形式で対称に表現できます。
- **符号反転が瞬時に完了**：数全体の符号を反転（−a）させるには、単にすべての桁の正負を反転（T ↔ 1）するだけで済みます。2の補数表現のように「全ビットを反転した上で最下位ビットに1を加算する」というキャリー伝搬の遅延が一切生じません。
- **丸め誤差の相殺**：数値を途中の桁で切り捨てた際の誤差期待値が厳密にゼロとなるため、デジタル信号処理やニューラルネットワーク演算において丸めバイアスが蓄積しません。

### 3.2 TritVerilogの文法体系

TritVerilogで記述可能な主要な演算子および構文と、それらが論理合成によってどのようなセル回路網へとマッピングされるかを下表にまとめます。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@6c56455e09d1ac06f73dfad8fc37c07e5210197f/notebook/assets/tables/syntax.svg" width="640" alt="syntax">

ビット幅の規則はVerilogに準拠しており、代入先の桁数に合わせて演算が行われます。オペランドの桁数が代入先より短い場合は上位桁がゼロ（0）で自動パディングされ、代入先に収まらない溢れた上位桁は切り捨てられます。

### 3.3 論理合成のパイプライン

TritFlowの論理合成エンジンは、Pythonで実装された決定論的な最適化コンパイラであり、以下の5段階のパイプラインを経てネットリストを生成します。

1. **フロントエンド解析と展開**：TritVerilogソースコードをAST（抽象構文木）へ構文解析し、モジュール階層、パラメータ定数（`parameter`）、および繰り返し生成文（`generate for`）をフラットな論理グラフへと展開します。`always @*` ブロック内の手続き的代入は、代入ごとに信号のバージョンを追従する静的単一代入（SSA）形式へと正規化され、`if` や `case` 等の条件分岐はマルチプレクサ（選択回路）へと展開されます。
2. **三値テクノロジマッピング**：演算子ごとに専用のセル構造へと置き換えます。加減算は実績ある全加算器・半加算器のキャリー連鎖回路へ、大小比較は上位桁からの減算・飽和判定回路へ、一致判定（`==`）や `case` ラベルは桁ごとの一致検出器へと写像します。
3. **論理最適化（三値ブール代数）**：定数の事前畳み込み（Constant Folding）、共通部分式の共有（CSE）、未使用ゲートの削除（Dead Code Elimination）を行います。さらに、二値のブール代数とは異なる、平衡三値特有の数学的恒等式を用いて式を簡単化します。
   - 二重反転：−(−x) = x
   - 冪等律：min(x, x) = x、max(x, x) = x
   - 境界値特性：min(x, +1) = x、max(x, +1) = +1、min(x, −1) = −1
   - NANYの双対性：NANY(x, 0) = −x
   - ド・モルガンの法則：min(−a, −b) = −max(a, b)、max(−a, −b) = −min(a, b)
4. **ファンアウト制限とバッファ木挿入**：後段の入力容量が過大になると信号の立ち上がり・立ち下がり時間（スルー）が劣化し、整定遅延が増大します。SPICEシミュレーションにより、1つの出力端子で同一セルを8個並列駆動しても出力電圧が正常レベルを維持できることを確認していますが、設計マージンを確保するため、論理合成では1セルあたりの最大ファンアウトを**4**に制限しています。ファンアウトが4を超えるネットには、バッファセル（BUF）による木構造（Buffer Tree）を自動挿入して負荷を均等分散します。
5. **形式的等価性検証（Formal Equivalence Checking）**：合成後のセル回路網が、元のHDL記述と数学的に100%等価であることを網羅的に検証します。

### 3.4 算術演算：加算（`+`）と乗算（`*`）の物理実装

1トリットどうしの加算では、和の取り得る範囲が (−1) + (−1) = −2 から (+1) + (+1) = +2 までの −2〜+2 となるため、桁上げ（キャリー）処理が不可欠です。たとえば、

1 + 1 = 2 = 3·1 + (−1)

となるため、当該桁の和（Sum）は −1、上位桁への桁上げ（Carry）は +1 になります。

三値全加算器（Full Adder）は、2つの入力 a, b と下位からの桁上げ c<sub>in</sub> に対し、a + b + c<sub>in</sub> = s + 3c<sub>out</sub> を満たす回路です。TritFlowの論理合成器は、HDLの `+` 演算子を、著者がテープアウトしたチップと完全に同一の回路トポロジー、すなわち**NANYゲート11個とINVゲート5個**からなる実績ある全加算器へと展開します（半加算器の場合はNANY 5個＋INV 2個）。多桁の加算器は、これらをリップルキャリー（桁上げ連鎖）で縦続接続することで構成されます。

In [ ]:
# 3-1 HDL：+ で書いた全加算器
full_add_source = """
module full_add_rtl(input trit a, b, cin, output trit sum, cout);
  assign {cout, sum} = a + b + cin;
endmodule
"""

In [ ]:
# 3-2 論理合成する
need("full_add_source", why="合成するHDLは、3-1に書いてあります。")
with step("論理合成") as s:
    full_add_rtl = flow.loads(full_add_source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in full_add_rtl.report()["cells"].items())
           + f" · 全{full_add_rtl.equivalence['vectors']}入力で等価")
bus_picker(full_add_rtl)

一方、1トリットどうしの乗算（`*`）は極めてシンプルです。取り得る値の範囲は常に −1・0・+1 のいずれかに収まり、**桁上げは一切発生しません**。重み w と入力 x の積は、w = +1 ならば x そのもの、w = −1 ならば符号反転 −x、w = 0 ならば 0 となります。論理合成器は、この乗算回路をテープアウト版と同一の**NAND 2個、NOR 1個、INV 1個**（計4ゲート）というコンパクトなセル網へと展開します。

In [ ]:
# 3-3 HDL：* で書いた1トリットの積
mul_source = """
module mul(input trit w, x, output trit p);
  assign p = w * x;
endmodule
"""

In [ ]:
# 3-4 論理合成する
need("mul_source", why="合成するHDLは、3-3に書いてあります。")
with step("論理合成") as s:
    mul = flow.loads(mul_source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in mul.report()["cells"].items())
           + f" · 全{mul.equivalence['vectors']}入力で等価")
bus_picker(mul)

### 3.5 条件分岐と選択回路：三値マルチプレクサの最適化

`if-else`、`case`、三項演算子 `?:` などの条件分岐は、条件信号に応じて複数の候補から1つの信号を出力へと導くマルチプレクサ（選択回路）へと合成されます。

二値論理では条件が0か1の2択であるため、選択回路は単純なAND-ORゲートで構成できます。しかし三値論理では、**条件信号そのものが「0（不定）」を取り得る**ため、条件が0のときに何を出力すべきかという仕様を明確に定義し、回路化する必要があります。TritFlowの論理合成器は、条件式の性質に応じて2種類の選択回路を賢く使い分けます。

以下のコードセルでは、同一の機能（op = +1 なら a を、op = −1 なら b を選択する）を異なる書き方で記述した例を比較します。

In [ ]:
# 3-5 HDL：同じ選択を2通りに書く
select_sources = {
"(op == 1't1) ? a : b": """
module sel_cmp(input trit a, b, op, output trit r);
  assign r = (op == 1't1) ? a : b;
endmodule
""",
"op ? a : b": """
module sel_raw(input trit a, b, op, output trit r);
  assign r = op ? a : b;
endmodule
""",
}

1. **二値的条件（Crisp Condition）：比較式 `op == 1't1`**
   比較演算の一致・不一致の結果を s と置くと、s は真（+1）か偽（−1）の2値しかとらず、決して中間値の0にはなりません。このように条件が0にならないことが保証される場合、出力 r は二値のAND-OR選択と数学的に同等な以下の式で極めて簡潔に構成できます。

$$r = \max\bigl(\min(s,\ a),\ \min(-s,\ b)\bigr)$$

   s = +1 のときは min(+1, a) = a、min(−1, b) = −1 となり、両者の max を取ることで a が出力されます。s = −1 のときは逆に対称となり b が出力されます。最小値（min）、最大値（max）、符号反転は、それぞれNAND、NOR、INVで最小限のセル数で合成可能です。
2. **一般三値条件：直接変数 `op`**
   条件として信号 `op` そのものを指定した場合、op = 0 のケースが物理的に存在します。TritVerilogの言語仕様では、条件が0のとき「a と b の値が一致していればその合意値を、異なっていれば中間値0を出力する（Kleeneの合意）」と定義しています。この動作を実現するには、条件の3値（−1・0・+1）それぞれに対応する入力タップを持った真の三値セレクタ（MUX3）が必要となります。

In [ ]:
# 3-6 論理合成して比べる
need("select_sources", why="合成するHDLは、3-5に書いてあります。")
with step("論理合成") as s:
    selects = {name: flow.loads(source).synthesize() for name, source in select_sources.items()}
    s.done(" · ".join(f"{name} {net.report()['total_cells']}セル" for name, net in selects.items()))
cell_counts(selects)
for net in selects.values():
    bus_picker(net)

結果を比較すると明らかなように、MUX3を必要とする2つ目の記述は、1つ目に比べてゲート数・遅延ともに大きくなります。TritFlowの論理合成エンジンは、ASTレベルで条件式が0を取らないことを静的に証明できた場合、自動的に軽量な第1のアーキテクチャへと最適化します。

### 3.6 等価性検証（Equivalence Verification）

論理合成の信頼性を担保するため、TritFlowは合成を実行するたびに、元のHDLの数学的意味と合成後のゲートネットリストが完全に一致することを自動検証します。HDLの振る舞い（高水準の整数・真理値評価）と、ゲートネットリスト（各セルの真理値表を用いた論理シミュレーション）を独立した2つの実行パスで評価し、すべての出力ビットを照合します。

入力ピン数が12トリット以下の場合は、全数網羅（3<sup>12</sup> = 531,441通り以下）で完全一致を数学的に保証します。12トリットを超える大規模回路については、再現可能なシードに基づく100,000パターンの疑似乱数テストベクトルを用いて厳格に検証します。ここまでの合成結果に表示されている「全○入力で一致」は、この検証をパスした証明です。

### 3.7 実践応用：三値ニューラルネットワークの内積積和回路（dot4）

冒頭で触れた[BitNet b1.58](https://doi.org/10.48550/arXiv.2402.17764)などの三値重みニューラルネットワークの核心は、三値重み w<sub>i</sub> ∈ {−1, 0, +1} と入力 x<sub>i</sub> との内積積和演算 y = Σ<sub>i</sub> w<sub>i</sub>x<sub>i</sub> の超並列実行にあります。著者がOpenSUSI-TR10でテープアウトした回路も、この1項分の積和ユニット（X + A × B + C<sub>in</sub> = S + 3C<sub>out</sub>）そのものです。

ここでは、3.4節で解説した乗算器と加算器を組み合わせ、4項の内積積和回路「**dot4**」をTritVerilogで記述し、論理合成してみましょう。各項の積 w<sub>i</sub>x<sub>i</sub> は −1〜+1、4項の和は −4〜+4 の範囲を取るため、出力 y は符号ビットなしのわずか2トリット（重み 3<sup>1</sup>, 3<sup>0</sup>）で完全に表現可能です。

In [ ]:
# 3-7 HDL：三値重みの4項の積和
dot4_source = """
module dot4(input trit w0, w1, w2, w3, x0, x1, x2, x3, output trit [1:0] y);
  assign y = w0*x0 + w1*x1 + w2*x2 + w3*x3;
endmodule
"""

In [ ]:
# 3-8 論理合成する
need("dot4_source", why="合成するHDLは、3-7に書いてあります。")
with step("論理合成") as s:
    dot4 = flow.loads(dot4_source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in dot4.report()["cells"].items())
           + f" · 全{dot4.equivalence['vectors']}入力で等価")

合成されたネットリストを見ると、4つの乗算ブロックがそれぞれNAND 2個、NOR 1個、INV 1個（計16ゲート）で構成され、それらの和を求める加算ツリーが全加算器1個と半加算器2個（NANY 21個、INV 9個）で無駄なく美しく構成されていることが確認できます。次章では、このdot4マクロを実際にシリコンチップ上に配置配線していきます。

## 4. 自動配置配線（Place & Route）

3章で論理合成したセルの接続情報（ネットリスト）をもとに、TR-1um PDKの幾何学的ルールに従ってセルを半導体シリコン上に最適配置し、2層のメタル配線で結線して最終的なGDSIIデータを生成します。本章では、3.7節で合成した4項の内積積和マクロ「**dot4**」を題材として、TritFlowの物理設計エンジンを見ていきます。

### 4.1 専用の配置配線エンジンを構築した理由

配置配線の問題自体は幾何学的な結線問題であるため、理論上は二値EDAツール（OpenROADやALIGNなど）を流用することも考えられます。実際にTR-1um PDKにおいても、LibreLane等のオープンソースEDAへの移植実験（[luyanaa/TR-1um](https://github.com/luyanaa/TR-1um)）が進められています。

しかし、既存の二値EDAツールをそのまま三値設計に適用するには、以下の3つの大きな技術的障壁が存在しました。

1. **配線層の制約（2層メタル）**：TR-1umはコストと製造性を重視した1 µmプロセスであり、配線に利用できるメタル層は**第1層（M1）と第2層（M2）のわずか2層**に限られます。近代的なOpenROAD等が前提とする4〜6層以上の多層配線スタックとは異なり、2層配線では電源供給と信号配線が同一平面上で激しく競合します。
2. **3本の電源レールとLiberty規格の限界**：すべての三値セルにはVDD（+5 V）、VMID（0 V）、VSS（−5 V）の3本の電源レールを隙間なく連続供給する必要があります。標準的なセルライブラリ仕様であるLiberty（.lib）は2電源（VDD/GND）の二値論理を前提としており、3本のレール構造や三値の遷移特性を直接記述することができません。
3. **三値特有の静的電流制約（IRドロップ対策）**：三値インバータや論理ゲートは、0 Vを出力する際に抵抗分圧による静的電流が流れ続けます。もし消費電流の大きなセルが1本のレール上に過度に集中すると、電源配線のエレクトロマイグレーション耐量を超過したり、配線抵抗による電圧降下（IRドロップ）によって0 Vの中間電位が変動してしまいます。

これらの問題を根本から解決し、高密度かつ確実にDRC/LVSクリーンを達成するため、TritFlowは「セル行の間に専用の配線チャネルを設けて2層で整然と配線する」決定論的な軽量P&Rエンジンを自前で開発しました。

### 4.2 配置アルゴリズム：信号流指向の初期配置と焼きなまし法

セルの配置は、横方向にセルが並ぶ「行（Row）」と、配線用の通り道である「チャネル（Routing Channel）」を交互に配置するチャネル型アーキテクチャを採用しています。

1. **トポロジカル初期配置**：回路の入力ピンから出力ピンに向かって、信号の流れ（トポロジカル順序）に沿って各セルを最も幅の短い行へと幅優先で順次詰めて配置します。これにより、初期状態から信号の向きが揃い、配線の交差が最小限に抑えられます。
2. **焼きなまし法（Simulated Annealing）による最適化**：初期配置を出発点として、600ステップの焼きなまし法によって配置を段階的に改善します。各ステップでは、以下の3つの摂動（変形操作）のいずれかを確率的に実行します。
   - **セルの交換（Swap）［確率 60%］**：2つのセルの位置を入れ替える。
   - **セルの移動（Move）［確率 30%］**：1つのセルを別の行や位置へと移す。
   - **セルの左右反転（Flip）［確率 10%］**：セルの向きを通常（R0）と左右反転（MY）で反転させ、ピンの位置を近接する配線に近づける。
3. **評価関数（コスト関数）と電流制約**：評価値（Energy）は、全ネットの端子座標を囲む外接矩形の半周長和（HPWL：Half-Perimeter Wire Length）を主成分とし、最も長い行の幅を小さな重みで加えたものです。ここで極めて重要なのが**電源レール電流の厳密なハード制約**です。各セルの静的電流値を行ごとに積算し、1行あたりの許容電流密度（0.25 mA/µm）を超えるような配置候補に対しては、コストを無限大として即座に棄却します。これにより、局所的な過電流の発生をアルゴリズム的に完全に封じ込めます。

### 4.3 配線アルゴリズム：グリッド迷路探索とリップアップ・再試行

配置が完了すると、配線格子（ピッチ 6 µm）上で信号線の自動配線を行います。配線の基本戦略は、**M1を横方向（水平チャネル内）**、**M2を縦方向（垂直方向・セル上の通過を許容）**とする直交2層配線です。

1. **A*迷路探索（Maze Routing）**：端子数が多く回路全体に広く跨がるネットから優先順位をつけて、A*アルゴリズムにより最短経路でビア（V1）を介して結線します。
2. **リップアップ＆リトライ（Rip-up and Reroute）**：後から配線するネットが既存の配線によってブロックされ結線不能（Routing Failure）に陥った場合、すべての配線を一旦取り外した上で、失敗したネットを最優先にして残りの配線順序をシャッフルし、最初から再配線します。この再試行を最大8回行います。
3. **チャネル拡張（Channel Expansion）**：8回の再試行でも配線が完了しない場合は、配線チャネルのトラック数を広げて配線空間を拡大し、再度配線を試みます（最大5回の拡張）。
4. **電源幹線の自動生成**：信号配線の完了後、両端に垂直なM2電源幹線を敷設し、各行のM1レールへと低抵抗ビアで確実に給電します。

下のコードセルを実行すると、配置配線が実行されます。配置の改善推移と、迷路配線が1本ずつ引かれていく様子をインタラクティブなアニメーションで振り返ることができます。

In [ ]:
# 4-1 配置配線する
need("dot4", why="配置配線する回路は、3-8で合成した積和です。")
with step("配置配線", view=LayoutView()) as s:
    dot4_layout = dot4.place_and_route(output_dir=BUILD/"dot4", seed=0)
    stats = dot4_layout.stats
    s.done(f"{stats['instance_count']}セル · {stats['row_count']}行 · {stats['die_size_um'][0]:.0f} × {stats['die_size_um'][1]:.0f} µm · "
           f"配線長の見積もり {stats['hpwl_before_um']/1e3:.1f} → {stats['hpwl_after_um']/1e3:.1f} mm · ビア {stats['via_count']}個",
           picture=replay(dot4_layout))

完成したdot4マクロのGDSレイアウトを以下に可視化します。各層（M1、M2、ビア、ポリシリコン、拡散層）を切り替えて内部構造を確認してください。

In [ ]:
# 4-2 GDS
need("dot4_layout", why="表示するGDSは、4-1で作ります。")
gds_viewer(dot4_layout.gds_path)

## 5. 物理検証・寄生抽出・三値タイミング解析

4章で自動生成されたdot4マクロのGDSレイアウトを、PDK公式の物理検証デッキで検査し、寄生容量を考慮したポストレイアウトSPICEシミュレーションによって実際の動作速度と波形を評価します。さらに、高精度なSPICE過渡解析と、高速な静的タイミング解析（STA）を三値回路向けに構築・照合します。題材は4章と同じ積和dot4です。

### 5.1 物理検証：公式デッキによるDRCとLVS

完成したレイアウトが東海理化のファブ製造規則を満たしているかを、OpenSUSI TR-1um PDK公式のKLayoutスクリプトを用いて検証します。

1. **DRC（Design Rule Check）**：最小配線幅、最小配線間隔、最小面積、エンクロージャ（囲みマージン）などの幾何ルール違反がないかを検査します。
2. **LVS（Layout Versus Schematic）**：GDSレイアウトからトランジスタの接続関係（ネットリスト）を抽出し、論理合成で得られたゲートネットリストと1対1で突き合わせます。素子のトポロジーだけでなく、電源レール（VDD/VMID/VSS）の物理的な導通性まで厳格に照合されます。

In [ ]:
# 5-1 DRCとLVS
need("dot4_layout", why="検証するGDSは、4-1で作ります。")
with step("DRC") as s:
    dot4_drc = dot4_layout.drc()
    s.done(f"違反 {dot4_drc['violations']}件")
with step("LVS") as s:
    dot4_lvs = dot4_layout.lvs()
    s.done("一致" if dot4_lvs["matched"] else "不一致")

### 5.2 寄生抽出（PEX）と配線容量の見積もり

現在のOpenSUSI TR-1um PDKには、サインオフ用の公式寄生抽出（PEX）デッキがまだ整備されていません。そこでTritFlowは、LVSで一致が証明されたGDSの配線幾何形状から、各ネットの基板に対する対地寄生容量（C<sub>gnd</sub>）を物理モデルに基づいて算出する抽出器を備えています。

抽出器は、第1層メタル（M1）および第2層メタル（M2）の多角形パターンから面積と周長を算出し、以下の物理パラメータを用いて配線容量を計算します。

- 絶縁膜（SiO<sub>2</sub>）の比誘電率：3.9
- 基板からの実効的な高さ：M1で 1 µm、M2で 2 µm
- 配線エッジのフリンジ（周辺）容量係数：両層共通で 0.02 fF/µm

本モデルではネット間の結合容量（クロストーク容量）や配線抵抗（寄生R）は省略し、ネットごとの集中容量負荷としてSPICEネットリストに付加しています。

In [ ]:
# 5-2 抽出と配線容量
need("dot4_lvs", why="抽出は、5-1でLVSが一致したGDSから行います。")
with step("抽出と配線容量") as s:
    dot4_extracted = dot4_layout.extract()
    dot4_wires = dot4_layout.result.net_parasitics()
    s.done(f"素子のネットリスト · 信号{len(dot4_wires['nets'])}本に配線容量を追加")
wire_table(dot4_wires)

### 5.3 三値回路における遅延の物理的定義

二値集積回路における伝搬遅延は、一般に出力電圧が電源電圧の半分（50%電位：V<sub>DD</sub>/2）を横切る時刻をもって測定されます。

しかし、**この従来の定義は三値回路では物理的に完全に破綻します**。なぜなら、電源電圧の中点である0 V（VMID）こそが、独立した有効な論理状態「0」そのものだからです。たとえば、出力が −1（−5 V）から +1（+5 V）へと大きくスイングする際、波形は過渡的に必ず中間の0 V付近を通過します。もし中点クロスを基準にしてしまうと、信号がまだ過渡遷移の途上にあるにもかかわらず「スイッチング完了」と誤判定してしまいます。また、出力が0 Vを目指して遷移する場合には、抵抗分圧回路の時定数に従って波形が指数関数的に0 Vへと漸近するため、中点クロスという概念自体が成立しません。

そこでTritFlowでは、制御工学やアナログIC設計の知見を取り入れ、遅延を**整定時間（Settling Time）**として厳密に定義しています。

> **三値における遅延の定義**：入力波形が変化を開始した時刻から、出力波形が最終目標電位（−5 V・0 V・+5 V）の許容判定帯（±0.5 V）内に最終的に突入し、二度とそこから外れなくなるまでの時間。

下のセルでは、内積積和回路dot4の重みを w = (+1, −1, +1, +1) に固定し、入力 x を桁上げ（キャリー）が次々と伝搬するテストベクトル列で駆動したときのポストレイアウトSPICE解析を実行します。これは、機械学習モデルの重みを固定して推論を実行する実運用環境と全く同一の動作条件です。シミュレーション波形上に、最も遅延の大きかった最悪経路の整定時間を矢印で示します。

In [ ]:
# 5-3 ポストレイアウトシミュレーション
need("dot4_extracted", why="シミュレーションする回路は、5-2で抽出します。")
from tritflow.sim.postlayout import logical_nodeset
dot4_inputs = [p.name for p in dot4.ports if p.dir == "input"]
weights = (+1, -1, +1, +1)
activations = [(0, 0, 0, 0), (1, 0, 0, 0), (1, -1, 0, 0), (1, -1, 1, 1), (-1, 1, -1, -1), (0, 0, 0, 0),
               (0, 1, 0, 0), (-1, 1, 0, 0), (1, -1, 1, 0), (-1, 1, -1, 0), (0, 0, 0, 0)]
def operands(x):
    """The circuit's input trits for the fixed weights and the activations x, in port order."""
    values = {f"w{i}": w for i, w in enumerate(weights)} | {f"x{i}": v for i, v in enumerate(x)}
    return tuple(values[p] for p in dot4_inputs)
dot4_vectors = [operands(x) for x in activations]
with step("ngspice") as s:
    dot4_post = dot4_extracted.simulate(dot4_vectors, load=100e-15, hold_s=1e-6, step_s=5e-9, edge_s=1e-9,
                                        temperature_c=27, timeout=600, workdir=BUILD/"dot4"/"postlayout",
                                        nodeset=logical_nodeset(dot4, dot4_lvs["report"], dot4_inputs, dot4_vectors[0]))
    s.done("全区間で ±0.5 V の帯に入った" if dot4_post["measurements"]["passed"] else "帯に入らない区間がある")
display(HTML(scope_figure(dot4_post, settle=True).to_html(include_plotlyjs="cdn", full_html=False,
                                                         config={"displaylogo": False, "scrollZoom": True})))
transient_conditions(dot4_post, load_f=100e-15)

### 5.4 三値静的タイミング解析（Ternary STA）

回路規模が増大すると、SPICEによる全パターンの過渡解析は膨大な計算時間を要するようになります。回路の最大動作周波数を瞬時に見積もるには、SPICEを実行せずに遅延を予測する**静的タイミング解析（STA：Static Timing Analysis）**が欠かせません。

二値STAで考慮すべき遷移は「立ち上がり（Rise）」と「立ち下がり（Fall）」の2種類のみですが、三値論理では取り得る論理状態が3つあるため、**計6種類の異なる状態遷移**（−1 → 0、−1 → +1、0 → −1、0 → +1、+1 → −1、+1 → 0）が存在し、セルの伝搬遅延は遷移ごとに異なります。

TritFlowの三値STAエンジンは、セルごとに「入力遷移の種類」「入力変化時間（スルー）」「出力負荷容量」を入力として、事前に高精度SPICEで網羅的にキャラクタライズした特性ルックアップテーブルを参照して動作します。

- **段間の中点伝搬時間（p<sub>50</sub>）**：各論理段のゲート内部における伝搬遅延は、入力中点（遷移前後の電位の中間）と出力中点との時間差として測定・合成されます。
- **終端整定テール時間（t<sub>tail</sub>）**：中点横切り時刻を足し合わせたのち、最終出力段において「中点横切りから ±0.5 V 判定帯へと完全に整定するまでの時間（整定テール）」を加算することで、5.3節で定義した物理整定時間と完全に整合させます。

以下に、入力スルーが1 nsにおけるINVセルの特性テーブルを可視化します。

In [ ]:
# 5-4 INVの遅延の表
delay_table("INV")

INVの入力は1本なので表は6行ですが、NANDやNANYなどの2入力セルでは、他方の入力の値ごとの表と、2入力が同時に遷移する場合の表を保持しています。5.3節と同じ入力ベクトル系列について、STAの見積もりとSPICEシミュレーションを比較します。

In [ ]:
# 5-5 三値のSTAとSPICEの比較
need("dot4_post", why="比べるSPICEの結果は、5-3で求めます。")
from tritflow.sta import analyze
states = [dict(zip(dot4_inputs, v)) for v in dot4_vectors]
with step("三値のSTA") as s:
    dot4_timing = analyze(dot4, vector_pairs=list(zip(states, states[1:])),
                          wire_caps={n: e["wire_capacitance_f"] for n, e in dot4_wires["nets"].items()},
                          output_load_f=100e-15, input_transition_s=1e-9)
    s.done(f"{len(dot4_timing.arrival)}出力")
timing_table(dot4_post, dot4_timing)
note("STAの条件：5-3と同じ入力の列、出力の負荷 100 fF、入力の変化時間 1 ns、配線容量は5-2の抽出値。")

## 6. 評価・応用・フルチップ統合

5章では、4項の内積積和マクロdot4を対象に、ポストレイアウトSPICEシミュレーションおよび三値STAによる物理検証を実施しました。本章ではまず、積和演算の規模（項数 N）を拡張した際のスケーリング特性（回路面積・伝搬遅延・静的電力）を体系的に評価します。続いて、HDLの記述変更が即座にシリコンレイアウトへと反映されるフローの俊敏性を実演し、最後に、実際の半導体シャトル製造に向けたMPWフレームおよびESD保護パッドを統合した最上位フルチップレイアウトを生成します。

### 6.1 内積積和回路の規模スケーリングと面積評価

評価対象として、三値の重みと入力による内積積和回路 y = w<sub>0</sub>x<sub>0</sub> + … + w<sub>N−1</sub>x<sub>N−1</sub> を取り上げます。項数を N = 2, 4, 8, 16 と変化させ、回路規模に対するスケーリング特性を調べます。

出力 y のトリット幅は、取り得る最小値 −N から最大値 +N までを表せる最小の桁数（N=2 で2トリット、N=4 で2トリット、N=8 で3トリット、N=16 で4トリット）とします。重みはHDLの入力として残し、シミュレーションでは (+1, −1, +1, +1) を周期的に繰り返した値に固定します。加算は3.7節と同じ書き方で合成し、項数に応じた構造の変更は加えません。

下のセルでは、各規模における自動配置配線および物理検証の記録を示します。面積は、配置配線したダイの外形寸法から算出されています。

In [ ]:
# 6-1 積和の規模と面積
from scale_views import load_scaling, scaling_table, synthesis_table
# 全掃引は長時間かかる（N=16の解析とN=32の配置配線試行を含む）。
RECOMPUTE_SCALING = False #@param {type:"boolean"}
dot_scaling = load_scaling(recompute=RECOMPUTE_SCALING)
display(synthesis_table(dot_scaling), scaling_table(dot_scaling))
for attempt in dot_scaling.get("optional_attempts", []):
    stage = "配置配線が未完了" if not attempt.get("pnr") else "検証が未完了"
    note(f"追加試行 N={attempt['n']} は{stage}のため、以下の比較には含めない。")
for row in dot_scaling["rows"]:
    options = row["conditions"].get("pnr_options", {})
    if options:
        note(f"N={row['n']}で既定値から変更した配置配線条件："
             f"行間チャネルの配線トラック {options['channel_tracks']}本、"
             f"M1の幅1 µm当たりの許容電流 {options['current_density_a_per_um']*1e3:g} mA。")
conditions = dot_scaling["conditions"]
holds = "、".join(f"N={row['n']}: {row['conditions']['hold_s']*1e6:g} µs"
                  for row in dot_scaling["rows"])
note(f"記録の条件：ngspice-{dot_scaling['provenance']['simulator']['version']}、"
     f"PDK {dot_scaling['provenance']['pdk']['qualified_commit'][:7]}、"
     f"{conditions['temperature_c']:g} °C、電源 +5 / 0 / −5 V。"
     f"出力負荷 {conditions['load_f']*1e15:g} fF、入力の変化時間 {conditions['edge_s']*1e9:g} ns、"
     f"最大時間刻み {conditions['step_s']*1e9:g} ns。各状態の保持時間は {holds}。")

項数 N が増えると、乗算ゲート（NAND, NOR, INV）が増加するだけでなく、和を表す桁数に応じて加算回路も増えます。さらにTR-1umの2層配線では、信号線数が増えるほど配線チャネルの必要トラック数が増加するため、ダイ面積はセル単体の面積総和よりも急峻に増加します。配置配線エンジンは、行数やチャネル幅を自動調整しながら、いずれの規模においてもDRC/LVSクリーンなレイアウトを出力します。

### 6.2 静的電力解析と時間平均消費電力

三値CMOS論理ゲートは、抵抗分圧によって0 Vを維持する状態において、入力を切り替えなくても静的電流が流れます。ただしNANYセルには、上下の直列MOS経路を遮断し、VMIDへのクランプによって0 Vを出力する状態もあり、このときは静的電流が流れません。

静的電力は、記録したテストベクトル系列に含まれる各状態のDC動作点からSPICEで厳密に求めています。正電源VDD（+5 V）および負電源VSS（−5 V）の双方から供給される電力を含め、回路全体の消費電力を算出しています。

時間平均の供給電力は、入力列の全保持区間にわたって電力波形を積分して求めています。この値には、入力状態ごとに異なる静的消費電力と、遷移時の動的スイッチング電力の両方が含まれます。入力を駆動する電圧源の電力は含めません。

In [ ]:
# 6-2 静的電流と電力
need("dot_scaling", why="表示する記録は、6-1で読み込みます。")
from scale_views import power_table, power_figure
power_plot = power_figure(dot_scaling)
power_plot.update_yaxes(title_text="供給電力 (mW)")
display(power_table(dot_scaling), power_plot)

### 6.3 三値STAとSPICE過渡解析の照合精度

各回路規模において、正負の桁上げの発生と解消を含む入力ベクトル系列を与え、固定した重みの条件の下で三値STAの予測遅延とngspiceの実測過渡遅延を比較します。STAとSPICEには同一の入力波形スルー（1 ns）、出力負荷容量（100 fF）、および抽出配線寄生容量を与えています。比較する遅延は、その系列で得られた出力ごとの最大整定時間です。

さらに、抽出された幾何形状から算出した**配線寄生容量の有無による遅延の影響**を比較評価します。素子の寄生容量と100 fFの出力負荷は両方に残し、推定した配線容量だけを除いた場合と比較することで、配線が回路速度に与える影響を定量化します。STAは全規模で、SPICEは4項で比較を行っています。

なお、N=16 の大規模回路では、内部配線により一部のNANYセルの負荷容量が事前特性テーブルの上限（2 pF）を超過し、最大 2.862 pF に達します。TritFlowの三値STAエンジンは、遅延を負荷容量について線形外挿して堅牢にタイミングを評価します。

In [ ]:
# 6-3 STAとSPICE
need("dot_scaling", why="比較する記録は、6-1で読み込みます。")
from scale_views import scaling_figure, layout_penalty_table
display(scaling_figure(dot_scaling), layout_penalty_table(dot_scaling))

比較結果が示すように、三値STAエンジンはSPICEのシミュレーション結果に対し、わずか数パーセントの極めて高精度な一致を達成しています。これにより、回路規模が拡大した場合でも、膨大なSPICE実行を繰り返すことなく、正確なクリティカルパスと最大動作速度を瞬時に特定できることが実証されました。

### 6.4 HDLの記述変更とフローの即時再実行

推論時の重みをHDLの定数として記述すると、重みに応じた乗算回路が論理合成で大幅に簡単化されます。以下のセルでは、dot4の重みを (+1, −1, +1, +1) に固定してTritVerilogを書き直しています。式と入出力の宣言を変更しても、論理合成から配置配線、DRC/LVS、寄生抽出、ポストレイアウトシミュレーションまでを同じセルで一貫して再実行できます。

In [ ]:
# 6-4 HDLを変更する
reuse_source = """
module dot4_fixed(input trit x0, x1, x2, x3, output trit [1:0] y);
  assign y = x0 - x1 + x2 + x3;
endmodule
"""

SPICEによる過渡解析では、全状態数が27以下であればすべての組み合わせを網羅し、それを超える場合はゼロ入力、全入力が−1または+1の状態、符号が交互に反転する状態、および1入力のみが±1へ振れる状態など、代表的な11状態を自動選択して検証します。出力の期待値は、合成された回路網からではなく、HDLの評価から直接算出して照合します。

In [ ]:
# 6-5 変更したHDLから全工程を実行する
need("reuse_source", why="実行するHDLは、6-4に書いてあります。")
from reuse import run
reuse_result = run(reuse_source, output_dir=BUILD/"reuse", step_s=5e-9,
                   exhaustive_limit=27, sample_size=11)

### 6.5 シャトル製造に向けたフルチップ統合（MPWトップレベル）

設計した三値コアマクロを実際の半導体ファブで製造するためには、チップ外周に入出力パッド（PAD）を配置し、静電気放電（ESD）保護素子を備えたファブ指定のMPW（Multi-Project Wafer）標準フレームと結合する必要があります。

本節では、dot4マクロを中心として、OpenSUSI TR-1umの公式MPWフレームと接続した最上位フルチップレイアウトを生成します。

ここで回路技術的に重要なのが、**ESD保護回路と三値電源の電圧整合**です。
TR-1umの標準入出力パッドに内蔵されたESD保護ダイオードは、通常 0 V 〜 +5 V（電圧レンジ 5 V）の二値信号を想定して設計されています。しかし、本三値回路の信号電圧は −5 V 〜 +5 V であり、全体の電位差は 10 V に達します。もしESD回路の基準電位を誤って0 V（VMID）に接続してしまうと、−5 V の信号が入力された瞬間にESD保護ダイオードが順方向に導通し、大電流が流れてチップが破壊されてしまいます。

そこでTritFlowでは、**フレームのESD保護の電源は、VSSを回路のVSS（−5 V）と共通にし、VDDには別の端子から+5 Vを与える**という接続アーキテクチャを採用しています。これにより、ESD保護素子の内部電位窓が対VSSで 10 V レンジに拡大され、−5 V 〜 +5 V の三値信号がESD保護の電源範囲内に完全に収まり、標準のパッドをそのまま安全に使えます。この接続方法は、2026年にテープアウトされた実チップ「OpenSUSI-TR10」の実証済み回路構成と完全に同一です。

In [ ]:
# 6-6 MPWフレームとパッドを付けたdot4
from reuse import shuttle_view
dot4_chip = shuttle_view(ROOT/"notebook/assets/chip/dot4")

### 6.6 結論

本ノートブックでは、日本の商用半導体ファブが公開したオープンソースPDK「OpenSUSI TR-1um」をターゲットとして、世界初となる三値集積回路のための完全自動化EDA設計フロー「**TritFlow**」を実証しました。

- **三値専用HDLからGDSIIまでの全自動パイプライン**：TritVerilogによる高水準記述から、三値ブール代数に基づく論理合成、決定論的な2層メタル配置配線、PDK公式デッキによるDRC/LVSクリア、寄生配線容量の抽出、そしてポストレイアウトSPICE解析までを一気通貫で完結させました。
- **物理・回路設計の深層的探求**：CMOSプロセス上で受動拡散抵抗の比精度によって中間電位0 Vを安定生成するINV回路の設計理論を確立し、Murmann流の手計算モデルとSPICE解析の高精度な一致を実証しました。また、Jerzy Słupeckiの判定法に基づき、NANYゲートと定数セルTIEPのみで任意の三値論理を合成可能な機能的完全性を証明しました。
- **三値固有のタイミングモデルとSTAの構築**：6種類の状態遷移と整定時間に基づく独自の遅延モデルを定義し、過渡SPICEシミュレーションと高い相関を持つ高速な三値静的タイミング解析エンジンを実現しました。
- **オープンソースエコシステムと実シリコンへの架橋**：本フローで用いたスタンダードセルは、2026年に実際にテープアウトされたシリコン設計と素子レベルで完全同一であり、2027年1月に納品される実チップの測定結果と直接比較照合されます。さらにMPWシャトル用の最上位フルチップレイアウトを自動生成することで、次期シャトルでの製造可能性を確立しました。

TritFlowは、BitNetに代表される次世代AIアーキテクチャや超低消費電力コンピューティングのための新たなハードウェア探索の扉を開きます。すべてのソースコード、セルライブラリ、および設計フローはApache-2.0ライセンスのもとで完全公開されており、半導体設計に関心を持つ世界中の研究者や学生が、自らの手で三値半導体の未来を切り拓くための強固なプラットフォームとなることを確信しています。

## 7. 対話的プレイグラウンド（Playground）

本章は、読者が自ら自由な三値回路を設計し、TritFlowの全自動EDAフローを体感できるインタラクティブな実験場です。

下のコードセル内の `TritVerilog` ソースコードを書き換えて実行するだけで、**論理合成 → セルマッピング → 2層メタル配置配線 → GDSII生成 → PDK公式DRC/LVS検証 → 寄生配線容量抽出 → ポストレイアウトSPICE過渡解析** の全工程が数秒から十数秒で自動実行され、回路図、レイアウト、およびアナログ波形が出力されます。

初期状態では、キャリー伝搬を伴う「2トリット加算器（`add2`）」の記述が設定されています。以下のような回路を自由に記述して、三値ハードウェア設計の可能性をぜひ探索してみてください。

- **三値マルチプレクサや条件分岐**：`op == 1't1 ? a : b` や `case` 文を用いたセレクタ
- **三値ALU（算術論理演算器）**：加算 `+`、減算 `-`、論理積 `&`、論理和 `|`、符号反転 `~` の複合回路
- **自作の三値関数回路**：NANYプリミティブや真理値表に基づく独自ロジック
- **多ビット積和演算器**：ビット幅や項数を変えた内積回路

In [ ]:
# 7-1 Playground
playground_source = """
module add2(input trit [1:0] a, b, output trit [2:0] s);
  assign s = a + b;
endmodule
"""
from reuse import run
playground = run(playground_source, output_dir=BUILD/"playground", step_s=5e-9,
                 exhaustive_limit=27, sample_size=11)

## 参考文献

1. Shuming Ma et al., “The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits,” arXiv:2402.17764, 2024. [doi:10.48550/arXiv.2402.17764](https://doi.org/10.48550/arXiv.2402.17764).
2. Sheng Lin, Yong-Bin Kim, and Fabrizio Lombardi, “CNTFET-Based Design of Ternary Logic Gates and Arithmetic Circuits,” *IEEE Transactions on Nanotechnology*, 10(2), pp. 217–225, 2011. [doi:10.1109/TNANO.2009.2036845](https://doi.org/10.1109/TNANO.2009.2036845).
3. Xiao-Yuan Wang et al., “High-Density Memristor-CMOS Ternary Logic Family,” *IEEE Transactions on Circuits and Systems I: Regular Papers*, 68(1), pp. 264–274, 2021. [doi:10.1109/TCSI.2020.3027693](https://doi.org/10.1109/TCSI.2020.3027693).
4. Jae Won Jeong et al., “Tunnelling-based ternary metal–oxide–semiconductor technology,” *Nature Electronics*, 2(7), pp. 307–312, 2019. [doi:10.1038/s41928-019-0272-8](https://doi.org/10.1038/s41928-019-0272-8).
5. Hanmok Park, Seonghoon Kim, Sunmean Kim, and Taigon Song, “An RTL-Based General Synthesis Methodology for Device-Independent Ternary Logic Circuits,” *IEEE Access*, 13, pp. 163648–163660, 2025. [doi:10.1109/ACCESS.2025.3597293](https://doi.org/10.1109/ACCESS.2025.3597293).
6. Steven Bos and contributors, *MixedRadixCircuitSynthesis (MRCS)*, source snapshot `0865cee`, 2023. [リポジトリ](https://github.com/aiunderstand/MixedRadixCircuitSynthesis/tree/0865cee1f49fe9a803b4910b570f4e96e514c61b).
7. Mohnish Pai, “Ankhdjet: An Open-Source Compiler for Mask-Programmed Ternary Compute-in-ROM on an Open PDK,” arXiv:2608.26206, 2026. [doi:10.48550/arXiv.2608.26206](https://doi.org/10.48550/arXiv.2608.26206).
8. Jerzy Słupecki, “Kryterium pełności wielowartościowych systemów logiki zdań,” *Comptes Rendus des Séances de la Société des Sciences et des Lettres de Varsovie, Classe III*, 32, pp. 102–109, 1939. [原文](https://rcin.org.pl/Content/50652/PDF/WA35_4821_cz167-r1939-t32_Spraw-TNW-wIII-art1.pdf#page=110).
9. Jean W. Butler, “On complete and independent sets of operations in finite algebras,” *Pacific Journal of Mathematics*, 10(4), pp. 1169–1179, 1960. [本文](https://msp.org/pjm/1960/10-4/pjm-v10-n4-p10-s.pdf).
10. Stephen Cole Kleene, *Introduction to Metamathematics*, North-Holland / P. Noordhoff, 1952, §64. [書誌情報](https://cir.nii.ac.jp/crid/1971712334708500134).
11. Francesco Paoli and Gavin St. John, “Editorial Introduction,” *Studia Logica*, 112, pp. 1201–1214, 2024. [doi:10.1007/s11225-024-10147-9](https://doi.org/10.1007/s11225-024-10147-9).
12. OpenSUSI, Tokai Rika, and contributors, *TR-1um Open Source PDK project*, commit `5d963db`, 2026. [使用したPDK](https://github.com/OpenSUSI/TR-1um/tree/5d963db9d44e192d37a0489c208bfccf2e2f690c).
13. Robin Lu and contributors, *TR-1um: an experiment to port TR-1um and OpenSUSI MPW template to LibreLane + ALIGN*, source snapshot `b14b3f5`, 2026. [リポジトリ](https://github.com/luyanaa/TR-1um/tree/b14b3f5cca671fd86804e3cca434a0a728c73642).
14. Einosuke Okazaki, *平衡3値論理回路 — OpenSUSI TR-1um (opensusi-tr10-balanced-ternary-logic)*, source snapshot `8a60b97`, 2026. [元の回路と提出資料](https://github.com/PenguinEino/opensusi-tr10-balanced-ternary-logic/tree/8a60b97868cad008ff21fefca5da13dcd06063f8).
15. 一般社団法人OpenSUSI, “OpenSUSI-MPW第1回募集を完了 — 企業・大学・高専・高校・個人から49枠の申し込み,” 2026年10月16日. [発表](https://prtimes.jp/main/html/rd/p/000000014.000182004.html).
16. Matthias Köfferlein and the KLayout contributors, *KLayout Layout Viewer and Editor*, version 0.30.6, 2026. [Release](https://github.com/KLayout/klayout/releases/tag/v0.30.6).
17. Holger Vogt, Giles Atkinson, and Paolo Nenzi, *Ngspice User's Manual*, version 42, 2023. [Manual](https://sourceforge.net/projects/ngspice/files/ng-spice-rework/old-releases/42/ngspice-42-manual.pdf/download).
18. ISHI会, “2025年12月イベント「東海理化シャトルのオープンPDK解説会」の参加者募集,” 2025年10月30日. [発表](https://ishi-kai.org/information/seminar/2025/10/30/DecEvent_1213.html).
19. Zane Hambly, *Takahe*, source snapshot `e7123ea`, 2026. [二値で符号化したSKY130の例](https://github.com/Zaneham/takahe/blob/e7123ea60269922acb283d576678a3aabe51f4f4/designs/setun70_physical.sv).
20. Nikolay P. Brusentsov and José Ramil Alvarez, “Ternary Computers: The Setun and the Setun 70,” in *Perspectives on Soviet and Russian Computing (SoRuCom 2011)*, IFIP AICT 357, Springer, pp. 74–80, 2011. [doi:10.1007/978-3-642-22816-2_10](https://doi.org/10.1007/978-3-642-22816-2_10)